In [ ]:
import psycopg2
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

def execute_query(query, columns_labels):
    try:
        # connect to a PostgresSQL database
        conn = psycopg2.connect(
        host="localhost",
        database="global_cybersecurity_threats",   # sepcify dataabse to connect to
        user="postgres",        # deafult user name
        password=__import__("os").environ["PGPASSWORD"])      # password
        
        # create a cursor
        cur = conn.cursor()
        
        # execute query
        cur.execute(query)
        
        # fetch the result and store into a python dataframe
        rows=cur.fetchall()

        result=pd.DataFrame(rows, columns=columns_labels)
        
        return result
    except (Exception, psycopg2.DatabaseError) as error:
        print(error)
    finally:
        if conn is not None:
            conn.close()        # close the communication with the PostgreSQL

In [ ]:
# create a sql query

query = '''SELECT distinct Country, Attack_Type
FROM threats '''

column_labels = ['Country', 'Attack Type']

result = execute_query(query, column_labels)

result.head()

In [ ]:
### Number of Attacks per Country

In [ ]:
# QUERY 1
query = ''' Select distinct Country, count(Attack_Type)
            from threats
            group by Country 
            order by count(Attack_Type) desc'''

column_labels = ['Country', 'Number of Attacks']

result = execute_query(query, column_labels)

result.head(10)

In [ ]:
plt.figure(figsize = (12,6))
plt.title('Number of Attacks in each Country')

sns.barplot(x = 'Number of Attacks', y = 'Country', data = result, palette = 'Blues_r')

In [ ]:
### Users affected per Attack Type

In [ ]:
# QUERY 2
query = ''' Select Attack_Type, sum(Number_of_Affected_Users)
            from threats
            group by Attack_Type
            order by sum(Number_of_Affected_Users) desc'''

column_labels = ['Attack Type', 'Number of Affected Users']

result = execute_query(query, column_labels)

result.head(6)

In [ ]:
plt.figure(figsize = (12,6))
plt.title('Number of Affected Users per Attack Type')

sns.barplot(x = 'Number of Affected Users', y = 'Attack Type', data = result, palette = 'Blues_r')

In [ ]:
### Financial Loss per Attack Type

In [ ]:
# QUERY 3
query = ''' Select Attack_Type, sum(Financial_Loss)
            from threats
            group by Attack_Type
            order by sum(Financial_Loss) desc'''

column_labels = ['Attack Type', 'Financial Loss']

result = execute_query(query, column_labels)

result.head(6)

In [ ]:
plt.figure(figsize = (12,6))
plt.title('Financial Loss (in Millions) per Attack Type')

sns.barplot(x = 'Financial Loss', y = 'Attack Type', data = result, palette = 'Blues_r')

#sns.scatterplot(x='Financial Loss', y='Attack Type', data=result, palette='Blues')

In [ ]:
### Country vs Attack Type: Are certain types of attacks more common in some countries than others

In [ ]:
# QUERY 4
query = ''' WITH Ranked_Attacks AS (SELECT Country, Attack_Type, COUNT(Attack_Type) AS Number_of_Occurrences,
           ROW_NUMBER() OVER (PARTITION BY Country ORDER BY COUNT(Attack_Type) DESC) AS Rank
           FROM threats
           GROUP BY Country, Attack_Type)

        SELECT Country, Attack_Type, Number_of_Occurrences
        FROM Ranked_Attacks
        WHERE Rank = 1
        ORDER BY Country; '''

column_labels = ['Country', 'Attack Type', 'Number of Occurrences']

result = execute_query(query, column_labels)
result.head(12)

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=result, x='Country', y='Number of Occurrences', hue='Attack Type', dodge=False)

plt.title('Most Common Attack Type per Country')
plt.xlabel('Country')
plt.ylabel('Number of Occurrences')
plt.legend(title='Attack Type', loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
### What are the most common types of threats and how has that changed over time?

In [ ]:
# QUERY 5
query = ''' Select distinct Attack_Type, count(Attack_Type)
            from threats
            group by Attack_Type 
            order by count(Attack_Type) desc'''

column_labels = ['Attack Type', 'Attack Occurence']

result = execute_query(query, column_labels)

result.head(10)

In [ ]:
gradient = sns.color_palette("Reds", n_colors=len(result))[::-1]

plt.figure(figsize = (10,4))
plt.title('Attack Type Occurences')
sns.barplot(x = 'Attack Occurence', y = 'Attack Type', data = result, palette = gradient)

In [ ]:
### Which industries are most targeted? (Top 3)

In [ ]:
# QUERY 6
query = ''' Select distinct Target_Industry, count(Attack_Type)
            from threats
            group by Target_Industry
            order by count(Attack_Type) desc'''

column_labels = ['Target Industry', 'Attack Occurence']

result = execute_query(query, column_labels)

result.head(3)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

colors = sns.color_palette('Set3', n_colors=len(result))

sizes = result['Attack Occurence']
labels = result['Target Industry']

plt.figure(figsize=(4, 4))
plt.title('Attack Occurrences by Target Industry')

plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=140, colors=colors)

plt.axis('equal') 
plt.show()

In [ ]:
### Which industries have the biggest financial losses?

In [ ]:
# QUERY 7
query = ''' Select distinct Target_Industry, count(Financial_Loss)
            from threats
            group by Target_Industry
            order by count(Financial_Loss) desc'''

column_labels = ['Target Industry', 'Financial Loss (in millions)']

result = execute_query(query, column_labels)

result.head(3)

In [ ]:
top_3 = result.head(3)

gradient = sns.color_palette("Reds", n_colors=len(top_3))[::-1]

plt.figure(figsize = (8,4))
plt.title('Financial Loss for Top 3 Target Industries')
sns.barplot(x = 'Target Industry', y = 'Financial Loss (in millions)', data = top_3, palette = gradient)

In [ ]:
import matplotlib.pyplot as plt

target_industries = result['Target Industry'][:3]
financial_losses = result['Financial Loss (in millions)'][:3]

def format_millions(values):
    total = sum(values)
    return ['{:.2f}M'.format(value) for value in values]

plt.figure(figsize=(6, 4))
plt.pie(financial_losses, labels=target_industries, startangle=140,
        autopct=lambda p: '{:.0f}M'.format(p * sum(financial_losses) / 100))
plt.title('Top 3 Target Industries by Financial Loss')
plt.show()